# Track 2: tokenización, stopwords, stemming, lematización y POS tagging

**Objetivo**: profundizar en cómo se prepara texto real —ruidoso, incompleto o normalizado a medias— después de Web Scraping u OCR.

**Qué vas a aprender**:
- tokenizar texto con NLTK y spaCy
- filtrar stopwords
- comparar stemming y lematización
- etiquetar partes de la oración (POS tagging)

**Salida esperada**: texto preparado para análisis posterior, búsqueda, clasificación o modelado.

**Dónde encaja**: este notebook va después de la adquisición del texto (scraping/OCR) y antes de la etapa de vectorización o análisis más avanzado.

> Track 2 no es "mejor" que Track 1: es más técnico y más profundo. Aquí pasamos de texto adquirido a texto preparado.


In [ ]:
# Celda única de preparación
# Si acabas de instalar paquetes o falla un import, reinicia el runtime y vuelve a ejecutar esta celda.
# Así cubrimos los recursos clásicos y los nombres nuevos que usan algunas versiones de NLTK.

import subprocess
import sys
from pprint import pprint

import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

def ensure_nltk_resource(resource_path, download_name):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name)

recursos_nltk = [
    ('tokenizers/punkt', 'punkt'),
    ('tokenizers/punkt_tab', 'punkt_tab'),
    ('corpora/stopwords', 'stopwords'),
    ('corpora/wordnet', 'wordnet'),
    ('corpora/omw-1.4', 'omw-1.4'),
    ('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger'),
    ('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng'),
]

for resource_path, download_name in recursos_nltk:
    ensure_nltk_resource(resource_path, download_name)

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp = spacy.load('en_core_web_sm')


## Preparación del texto base

Antes de tokenizar, limpiamos el texto para reducir ruido superficial: minúsculas, dígitos, puntuación y espacios repetidos.


In [ ]:
import re
import string

corpus_original = "Need to finalize the demo corpus which will be used for this notebook and it should be done soon !!. It should be done by the ending of this month. But will it? This notebook has been run 4 times !!"
corpus = corpus_original.lower()
corpus = re.sub(r'\d+', '', corpus)
corpus = corpus.translate(str.maketrans('', '', string.punctuation))
corpus = ' '.join(corpus.split())

print('Texto original:')
print(corpus_original)
print('\nTexto preparado:')
print(corpus)


## Tokenización

La tokenización separa el texto en unidades manejables. Primero miramos el corte básico; después usamos esas piezas en las demás técnicas.


In [ ]:
tokenized_corpus_nltk = word_tokenize(corpus)
doc = nlp(corpus)
tokenized_corpus_spacy = [token.text for token in doc]

print('Tokenización con NLTK:')
print(tokenized_corpus_nltk)

print('\nTokenización con spaCy:')
print(tokenized_corpus_spacy)


## Stopwords

Las stopwords son palabras muy frecuentes que suelen aportar poco significado por sí solas. Ojo: NLTK y spaCy no usan exactamente el mismo listado.


In [ ]:
stop_words_nltk = set(stopwords.words('english'))
tokens_without_sw_nltk = [token for token in tokenized_corpus_nltk if token not in stop_words_nltk]

doc = nlp(corpus)
stop_words_spacy = nlp.Defaults.stop_words
tokens_without_sw_spacy = [token.text for token in doc if not token.is_stop]

print('Tokens sin stopwords con NLTK:')
print(tokens_without_sw_nltk)

print('\nTokens sin stopwords con spaCy:')
print(tokens_without_sw_spacy)

print('\nDiferencias entre ambos enfoques:')
print(sorted(set(tokens_without_sw_nltk).symmetric_difference(set(tokens_without_sw_spacy))))


## Stemming

El stemming recorta palabras a una raíz aproximada. Es rápido y útil, pero puede producir formas no lingüísticas.


In [ ]:
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in tokenized_corpus_nltk]

print('Antes:')
print(tokenized_corpus_nltk)
print('\nDespués de stemming:')
print(stemmed_tokens)


## Lematización

La lematización busca el lema o forma base real de la palabra. Suele ser más precisa que el stemming, aunque depende más del contexto y del etiquetado gramatical.


In [ ]:
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokenized_corpus_nltk]

print('Tokens:')
print(tokenized_corpus_nltk)
print('\nLemas básicos:')
print(lemmatized_tokens)


## POS tagging

El POS tagging etiqueta la función gramatical de cada token. Esto ayuda a entender mejor la estructura de la oración y mejora tareas posteriores.


In [ ]:
print('POS tagging con spaCy:')
doc = nlp(corpus_original)
for token in doc:
    print(f'{token.text}: {token.pos_}')

print('\nPOS tagging con NLTK:')
pprint(nltk.pos_tag(word_tokenize(corpus_original)))


## Comparación: ¿cuándo usar cada técnica?

| Técnica | Qué hace | Cuándo conviene |
|---|---|---|
| Tokenización | Divide el texto en unidades | Siempre, como paso inicial |
| Stopwords | Quita palabras muy frecuentes y poco informativas | Cuando quieres reducir ruido |
| Stemming | Reduce palabras a una raíz aproximada | Búsqueda rápida, prototipos, normalización agresiva |
| Lemmatización | Devuelve la forma base correcta | Cuando importa más la calidad lingüística |
| POS tagging | Etiqueta la categoría gramatical | Cuando el contexto sintáctico importa |

Regla práctica: si necesitas velocidad, stemming puede bastar; si necesitas precisión lingüística, prioriza lematización y POS tagging.


## Atribución e inspiración

Este material está inspirado en *Practical Natural Language Processing* (O’Reilly), en https://www.practicalnlp.ai/ y en https://github.com/practical-nlp/practical-nlp-code/blob/master/Ch2/.

Se trata de una adaptación curada con fines de aprendizaje. No existe afiliación oficial con esos recursos.


## Del texto adquirido al texto preparado

Después de Web Scraping u OCR, el texto casi nunca llega listo para trabajar: trae ruido, símbolos, errores de captura, repeticiones y palabras que no agregan mucho valor. Por eso esta etapa existe.

Primero lo dividimos en tokens. Después quitamos stopwords para bajar ruido. Luego probamos stemming y lematización para normalizar variantes de una misma palabra. Finalmente, POS tagging nos ayuda a entender qué papel juega cada token dentro de la oración.

El resultado no es un texto "más bonito"; es un texto más útil. Y ese es el objetivo real: pasar del texto adquirido al texto preparado para análisis, búsqueda o modelado.
